Language: **English (default)** | [Polski](lab.ipynb)

# Lab 15 -- a local inference service with monitoring (EuroSAT + ResNet18)

Companion material to [`README.md`](README.md). **Before running this
notebook**: fill in the TODOs in `app/model.py`, `app/server.py`,
`benchmark.py`, `monitoring.py` (see README sections 2-3, 5-6), and run
`prepare_artifact.py` if you haven't already.

In [ ]:
import io  # noqa: F401 - used in the TODO below
import json  # noqa: F401 - used in the TODO below
import sys
import time  # noqa: F401 - used in the TODO below

import matplotlib.pyplot as plt  # noqa: F401 - used in the TODO below
import torch
import torchvision.datasets as tvd
from fastapi.testclient import TestClient
from torchvision.models import ResNet18_Weights

sys.path.insert(0, ".")

torch.manual_seed(0)
torch.set_num_threads(4)
device = torch.device("cpu")
print(f"Using device: {device}")

weights = ResNet18_Weights.IMAGENET1K_V1
preprocess = weights.transforms()
full_ds = tvd.EuroSAT(root="data_cache", download=True, transform=preprocess)
CLASS_NAMES = full_ds.classes
print(f"EuroSAT: {len(full_ds)} images, classes: {CLASS_NAMES}")

## 1. Model artifact -- consistency verification

`prepare_artifact.py` is provided and should already have been run
(`uv run python labs/15_production/prepare_artifact.py`).

In [ ]:
# TODO:
# 1. from app import model as model_module
# 2. loaded = model_module.load_artifact("artifacts")
# 3. pick one test image (PIL, NOT from full_ds[i][0] which already has
#    preprocessing applied -- you need the raw image to encode as PNG)
# 4. encode it to PNG bytes (io.BytesIO + img.save(..., format="PNG"))
# 5. compute the prediction TWO ways: model_module.predict(loaded, png_bytes)
#    and "by hand" (preprocess + forward + softmax) -- compare, they should
#    be identical (this is NOT a test through the API yet -- it's just
#    your own app/model.py code called twice)
raise NotImplementedError("TODO: artifact consistency verification")

## 2-3. `app/model.py` and `app/server.py`

Filled in **outside this notebook**, in the `app/model.py` and
`app/server.py` files (README sections 2-3). Below: building a client and
checking that the endpoints respond.

In [ ]:
from app.server import app as fastapi_app

with TestClient(fastapi_app) as client:
    r_health = client.get("/health")
    r_ready = client.get("/ready")
    r_info = client.get("/model-info")
    print(f"GET /health: {r_health.status_code} {r_health.json()}")
    print(f"GET /ready: {r_ready.status_code} {r_ready.json()}")
    print(f"GET /model-info: {r_info.status_code} {r_info.json()}")

## 4. API consistency with a direct call -- your task

In [ ]:
# TODO:
# 1. pick a real EuroSAT image (PIL, raw, no preprocessing applied),
#    encode it to PNG bytes
# 2. send POST /predict through `client` (use "with TestClient(fastapi_app) as client:")
# 3. compute model_module.predict(loaded, png_bytes) directly on THE SAME bytes
# 4. compare -- they MUST match EXACTLY (the same class and confidence)
raise NotImplementedError("TODO: API vs direct-call consistency")

In [ ]:
# TODO: check error handling:
# 1. a corrupted file (e.g. b"not an image") -> what HTTP code and message?
# 2. a file too large (e.g. b"0" * 6_000_000) -> what HTTP code?
# 3. make sure the message does NOT contain a full traceback/local paths
raise NotImplementedError("TODO: input error handling")

## 5. Benchmark: latency and throughput -- your task

`send_requests` is provided. `compute_stats` (in `benchmark.py`) is your
task (README section 5).

In [ ]:
from benchmark import compute_stats, send_requests  # noqa: F401 - used in the TODO below

# TODO:
# 1. encode one real EuroSAT image to PNG bytes (as above)
# 2. from `with TestClient(fastapi_app) as client:`, measure COLD (the 1st
#    request, separately) and WARM (20-30 consecutive requests, via send_requests)
# 3. compute_stats(warm_times) -- print mean/p50/p95/throughput
# 4. comment: how does cold compare to warm? (see the lecture for context)
raise NotImplementedError("TODO: cold/warm benchmark")

## 6. PSI -- verification on a known example

Before using `psi()` (your implementation in `monitoring.py`) on real
data, verify it on a known example from the lecture.

In [ ]:
from monitoring import histogram_counts, psi  # noqa: F401 - used in the TODO below

# TODO:
# ref = torch.tensor([50.0, 50.0]); cur = torch.tensor([70.0, 30.0])
# expected result: 0.169460 (from lecture.md) -- verify your psi()
# ALSO check: psi(ref, ref) should give 0 (identical distributions)
# ALSO check: an empty bin (e.g. cur=[100.0, 0.0]) -- with your eps, is the
# result finite (not inf/nan)?
raise NotImplementedError("TODO: PSI verification on a known example")

## 7. Monitoring scenarios A/B/C -- your task

See README section 7 for the full instructions. **IMPORTANT**: choose the
PSI threshold and the transformation strength BEFORE checking accuracy in
scenario C, not after.

In [ ]:
# Scenario A: control (the same distribution). Data already split below.
torch.manual_seed(0)
perm = torch.randperm(len(full_ds))
test_idx = perm[2200:2600]
control_idx = perm[2600:3000]


def load_subset(indices):
    X = torch.stack([full_ds[i][0] for i in indices])
    y = torch.tensor([full_ds[i][1] for i in indices])
    return X, y


X_test, y_test = load_subset(test_idx.tolist())
X_control, y_control = load_subset(control_idx.tolist())

# TODO:
# 1. compute brightness_feature(X_test) as the reference, build bin_edges
# 2. compute histogram_counts for brightness_feature(X_control)
# 3. psi(ref_counts, control_counts) -- expected result: LOW (no alarm)
raise NotImplementedError("TODO: scenario A -- control")

In [ ]:
# Scenario B: a controlled change (brightness OR contrast -- choose ONE,
# one strength, decided BEFORE seeing accuracy in scenario C)
# TODO:
# 1. build shifted_X (e.g. (X_control + SHIFT).clamp(X_control.min(), X_control.max()+1.0))
# 2. compute PSI(ref, shifted) -- does it detect the change?
raise NotImplementedError("TODO: scenario B -- a controlled change")

In [ ]:
# Scenario C: delayed labels -- you have y_control, so you can "simulate"
# their later availability
# TODO:
# 1. compute the model's accuracy on X_control (reference) and on shifted_X
#    (with THE SAME y_control)
# 2. compare with the PSI result from scenario B -- do drift and accuracy go
#    in the same direction?
# 3. do NOT assume the answer in advance -- write what ACTUALLY HAPPENED in
#    your experiment
raise NotImplementedError("TODO: scenario C -- effect on quality")

## 8. Diagnostic task: training-serving skew

`model_colleague.py` implements preprocessing **differently** than your
`app/model.py`. See README section 8.

In [ ]:
from model_colleague import (
    predict_colleague,  # noqa: F401 - used in the TODO below
    preprocess_image_bytes_colleague,  # noqa: F401
)

print("model_colleague.py imported -- compare with your own implementation.")

In [ ]:
# TODO:
# 1. on several (e.g. 10-20) real EuroSAT images, compute predict() (yours)
#    and predict_colleague() (the colleague's) for THE SAME bytes
# 2. how many predictions DIFFER (not just the logits)?
# 3. in a comment: exactly WHICH preprocessing step differs, and why it
#    matters (refer to ResNet18_Weights.IMAGENET1K_V1.transforms())
raise NotImplementedError("TODO: diagnosing training-serving skew")

## 9. Report: a decision on deploying and maintaining the model -- your task

**1. Model version, data source, the offline evaluation protocol.**

_(Your answer here.)_

**2. Accuracy and the most important limitations.**

_(Your answer here.)_

**3. The API's input/output format.**

_(Your answer here.)_

**4. Latency p50/p95, throughput (your measured numbers).**

_(Your answer here.)_

**5. The results of scenarios A/B/C.**

_(Your answer here.)_

**6. The level of confidence in the drift interpretation (one feature, a limited sample).**

_(Your answer here.)_

**7. Known risks, recommended next steps.**

_(Your answer here.)_